# Phase 4 — Full Evaluation

This notebook summarises the Phase 4 evaluation of the PubMed GraphRAG pipeline:

1. **Dataset statistics** — how many PubMedQA questions matched our 5,000-abstract subset.
2. **Retrieval comparison** — vector-only baseline vs. graph-enhanced (GraphRAG) retrieval.
3. **Generation quality** — ROUGE-L and BERTScore distributions for generated answers.
4. **Examples** — sample questions, references, and generated answers.

Run `scripts/run_evaluation.py` first to generate the output CSV/JSON files used below.

In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

OUTPUT_DIR = Path("outputs")
DATA_DIR = Path("data/evaluation")

## 1. Dataset statistics

In [ ]:
questions_path = DATA_DIR / "pubmedqa_filtered.jsonl.gz"
questions = [json.loads(line) for line in gzip.open(questions_path, "rt")]

print(f"Total PubMedQA questions matched to our subset: {len(questions)}")

match_scores = [q.get("match_score", 0.0) for q in questions]
print(f"Match-score min: {min(match_scores):.3f}")
print(f"Match-score max: {max(match_scores):.3f}")
print(f"Match-score mean: {np.mean(match_scores):.3f}")
print(f"Match-score median: {np.median(match_scores):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(match_scores, bins=20, edgecolor="black")
ax.set_xlabel("Query-to-abstract overlap similarity")
ax.set_ylabel("Number of questions")
ax.set_title("Distribution of article-match scores")
plt.show()

## 2. Retrieval comparison

In [ ]:
retrieval_results = pd.read_csv(OUTPUT_DIR / "retrieval_results.csv")
retrieval_summary = json.load(open(OUTPUT_DIR / "retrieval_summary.json"))

comparison = pd.DataFrame.from_dict(retrieval_summary, orient="index")
comparison = comparison[["recall@5", "recall@10", "mrr", "num_questions"]]
comparison

In [ ]:
metrics = ["recall@5", "recall@10", "mrr"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, comparison.loc["vector_only", metrics], width, label="Vector-only")
ax.bar(x + width/2, comparison.loc["graph_rag", metrics], width, label="GraphRAG")
ax.set_ylabel("Score")
ax.set_title("Retrieval metrics: vector-only vs GraphRAG")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, max(1.0, comparison[metrics].values.max() * 1.2))
plt.show()

### Per-question metric distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["recall@5", "recall@10", "mrr"]):
    for method, color in [("vector_only", "C0"), ("graph_rag", "C1")]:
        subset = retrieval_results[retrieval_results["method"] == method][metric]
        ax.hist(subset, bins=20, alpha=0.6, label=method, color=color, edgecolor="black")
    ax.set_xlabel(metric)
    ax.set_ylabel("Number of questions")
    ax.set_title(f"{metric} distribution")
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Generation quality

In [ ]:
generation_path = OUTPUT_DIR / "generation_results.csv"
if generation_path.exists():
    gen_results = pd.read_csv(generation_path)
    print(f"Generation records: {len(gen_results)}")
    print(gen_results[["rouge_l", "bertscore_f1"]].describe())
else:
    print("No generation results found. Run without --retrieval-only to generate answers.")

In [ ]:
if generation_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(gen_results["rouge_l"], bins=20, edgecolor="black")
    axes[0].set_xlabel("ROUGE-L F1")
    axes[0].set_ylabel("Number of questions")
    axes[0].set_title("ROUGE-L distribution")

    axes[1].hist(gen_results["bertscore_f1"], bins=20, edgecolor="black")
    axes[1].set_xlabel("BERTScore F1")
    axes[1].set_ylabel("Number of questions")
    axes[1].set_title("BERTScore F1 distribution")
    plt.tight_layout()
    plt.show()

## 4. Example generated answers

In [ ]:
if generation_path.exists():
    for _, row in gen_results.head(3).iterrows():
        print("=" * 80)
        print(f"Q: {row['question']}")
        print(f"Reference: {row['reference_answer'][:300]}...")
        print(f"Generated: {row['generated_answer'][:300]}...")
        print(f"ROUGE-L: {row['rouge_l']:.3f} | BERTScore F1: {row['bertscore_f1']:.3f}")
        print()